# Landscape Energetics from LiDAR-Derived Tree Census

This notebook implements the analytical workflow for estimating landscape-scale energy content from airborne LiDAR data at Gordon Gulch watershed, CO.

**Workflow steps:**
1. Load and inspect LiDAR point cloud (DEM / canopy height model)
2. Individual tree segmentation
3. Calculate tree dimensions (height, crown diameter)
4. Allometric biomass estimation (Jucker et al. 2017)
5. Energy content calculation (Higher Heating Value)
6. Landscape energy census and rasterization

## Setup and Dependencies

In [ ]:
# Install dependencies if needed
# !pip install rasterio numpy pandas matplotlib scipy laspy[lazrs]

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.plot import show
from rasterio.transform import from_bounds
import matplotlib.pyplot as plt
from pathlib import Path
import laspy
import os
from scipy.ndimage import maximum_filter, gaussian_filter, distance_transform_edt, uniform_filter
from scipy.stats import binned_statistic_2d

%matplotlib inline

## Configuration

Set paths to input DEM and LiDAR data. Update these paths when data locations are provided.

In [ ]:
# === DATA PATHS ===
DEM_PATH = "../data/gordon_gulch/gordongulch_dem_10m_pitremoved.tif"
LIDAR_DIR = "../data/gordon_gulch/lidar"  # Directory containing LAZ tiles
CHM_PATH = "../data/gordon_gulch/gordongulch_chm_05m.tif"
TREE_CSV_PATH = "../data/gordon_gulch/gordongulch_tree_census_05m.csv"

OUTPUT_DIR = Path("../output/energetics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_AREA_KM2 = 2.6  # km²
HHV_MEAN = 20.25  # MJ/kg, mean for Pinus spp.
RES = 0.5  # output raster resolution (m)

## Step 1: Load and Inspect Input Data

In [ ]:
# Load and display the DEM
with rasterio.open(DEM_PATH) as src:
    dem = src.read(1)
    dem_meta = src.meta
    dem_bounds = src.bounds
    dem_crs = src.crs
    print(f"CRS: {dem_crs}")
    print(f"Bounds: {dem_bounds}")
    print(f"Resolution: {src.res}")
    print(f"Shape: {dem.shape}")
    print(f"Elevation range: {np.nanmin(dem):.1f} - {np.nanmax(dem):.1f} m")
    print(f"NoData: {src.nodata}")
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    show(src, ax=ax, title="Gordon Gulch DEM (10m)")
    plt.colorbar(ax.images[0], ax=ax, label="Elevation (m)")
    plt.show()

## Step 2: Generate Canopy Height Model (CHM)

Read LAZ tiles from USGS 3DEP (CO_DRCOG_2020_B20), clip to DEM extent, and generate a 0.5m CHM by differencing the Digital Surface Model (DSM, max return) and Digital Terrain Model (DTM, ground-classified returns). A 3x3 mean filter is applied to both DSM and DTM for pit filling before differencing.

**Data source:** USGS 3DEP, CO DRCOG 2020 collection (6 tiles, ~818 MB LAZ)

In [ ]:
# Read DEM bounds for clipping
with rasterio.open(DEM_PATH) as src:
    dem_crs = src.crs
    dem_bounds = src.bounds

x_min, x_max = dem_bounds.left, dem_bounds.right
y_min, y_max = dem_bounds.bottom, dem_bounds.top
cols = int(np.ceil((x_max - x_min) / RES))
rows = int(np.ceil((y_max - y_min) / RES))
print(f"Output grid at {RES}m: {cols} x {rows} ({cols*rows:,} cells)")

# Read all LAZ tiles, clip to DEM extent
all_x, all_y, all_z = [], [], []
all_x_g, all_y_g, all_z_g = [], [], []

for f in sorted(os.listdir(LIDAR_DIR)):
    if not f.endswith('.laz'):
        continue
    print(f"Reading {f}...")
    with laspy.open(os.path.join(LIDAR_DIR, f)) as las_file:
        for pts in las_file.chunk_iterator(2_000_000):
            x, y, z = pts.x, pts.y, pts.z
            mask = (x >= x_min) & (x <= x_max) & (y >= y_min) & (y <= y_max)
            if mask.sum() == 0:
                continue
            xc, yc, zc = x[mask], y[mask], z[mask]
            cls = pts.classification[mask]
            all_x.append(xc); all_y.append(yc); all_z.append(zc)
            gm = cls == 2  # Ground-classified points
            if gm.sum() > 0:
                all_x_g.append(xc[gm]); all_y_g.append(yc[gm]); all_z_g.append(zc[gm])

all_x = np.concatenate(all_x); all_y = np.concatenate(all_y); all_z = np.concatenate(all_z)
all_x_g = np.concatenate(all_x_g); all_y_g = np.concatenate(all_y_g); all_z_g = np.concatenate(all_z_g)
print(f"\nTotal clipped points: {len(all_x):,}")
print(f"Ground points: {len(all_x_g):,}")

# Rasterize DSM (max z) and DTM (mean ground z)
x_edges = np.arange(x_min, x_min + (cols + 1) * RES, RES)[:cols + 1]
y_edges = np.arange(y_min, y_min + (rows + 1) * RES, RES)[:rows + 1]

print("Building DSM...")
dsm = binned_statistic_2d(all_x, all_y, all_z, statistic='max', bins=[x_edges, y_edges]).statistic.T[::-1].astype(np.float32)

print("Building DTM...")
dtm = binned_statistic_2d(all_x_g, all_y_g, all_z_g, statistic='mean', bins=[x_edges, y_edges]).statistic.T[::-1].astype(np.float32)

# Fill NaN gaps with nearest neighbor
for name, arr in [('DTM', dtm), ('DSM', dsm)]:
    nan_mask = np.isnan(arr)
    print(f"{name} NaN cells: {nan_mask.sum():,}")
    if nan_mask.any():
        indices = distance_transform_edt(nan_mask, return_distances=False, return_indices=True)
        if name == 'DTM':
            dtm = dtm[tuple(indices)]
        else:
            dsm = dsm[tuple(indices)]

# 3x3 mean filter for pit filling on DTM and DSM
print("Applying 3x3 smoothing filter for pit filling...")
dtm_smooth = uniform_filter(dtm, size=3)
dsm_smooth = uniform_filter(dsm, size=3)

# Only fill pits: replace where original is lower than smoothed neighborhood
dtm_pits = dtm < dtm_smooth
dtm[dtm_pits] = dtm_smooth[dtm_pits]
print(f"DTM pits filled: {dtm_pits.sum():,}")

dsm_pits = dsm < dsm_smooth
dsm[dsm_pits] = dsm_smooth[dsm_pits]
print(f"DSM pits filled: {dsm_pits.sum():,}")

# CHM = DSM - DTM
chm = dsm - dtm
chm[np.isnan(chm)] = 0
chm[chm < 0] = 0
chm[chm > 60] = 0

print(f"\nCHM range: {chm.min():.1f} - {chm.max():.1f} m")
print(f"Canopy coverage (>2m): {(chm > 2).sum() / chm.size * 100:.1f}%")
print(f"Mean canopy height (>2m): {chm[chm > 2].mean():.1f} m")

# Save rasters
transform_out = from_bounds(x_min, y_min, x_max, y_max, cols, rows)
out_meta = {'driver': 'GTiff', 'dtype': 'float32', 'width': cols, 'height': rows,
            'count': 1, 'crs': dem_crs, 'transform': transform_out, 'nodata': -9999,
            'compress': 'lzw'}

for name, data in [('chm_05m', chm), ('dsm_05m', dsm), ('dtm_05m', dtm)]:
    out_path = f"../data/gordon_gulch/gordongulch_{name}.tif"
    d = data.copy()
    d[np.isnan(d)] = -9999
    with rasterio.open(out_path, 'w', **out_meta) as dst:
        dst.write(d, 1)
    print(f"Saved {out_path}")

## Step 3: Individual Tree Segmentation

Variable-window local maxima detection on the CHM following Swetnam and Falk (2014). Window size scales with tree height (taller trees have wider crowns), reducing commission errors from the fixed-window approach.

In [ ]:
# Load CHM (or use from previous step if already in memory)
with rasterio.open(CHM_PATH) as src:
    chm = src.read(1)
    chm_transform = src.transform
    chm_crs = src.crs
    chm_meta = src.meta.copy()

chm[chm == -9999] = 0

# Smooth CHM to reduce spurious maxima
# sigma=2 at 0.5m resolution ≈ sigma=1 at 1m (1m effective smoothing kernel)
chm_smooth = gaussian_filter(chm, sigma=2.0)

# Variable-window local maxima detection
# Window sizes in pixels (at 0.5m, double the metric window size)
tree_tops = np.zeros_like(chm, dtype=bool)

height_window_map = [
    (3, 6, 6),     # 3m window → 6 px at 0.5m
    (6, 10, 10),   # 5m → 10 px
    (10, 15, 14),  # 7m → 14 px
    (15, 20, 18),  # 9m → 18 px
    (20, 35, 22),  # 11m → 22 px
]

for h_min, h_max, win_size in height_window_map:
    local_max = maximum_filter(chm_smooth, size=win_size) == chm_smooth
    height_mask = (chm_smooth >= h_min) & (chm_smooth < h_max)
    tree_tops |= (local_max & height_mask)

# Remove edge artifacts (4 px = 2m at 0.5m resolution)
tree_tops[:4, :] = False
tree_tops[-4:, :] = False
tree_tops[:, :4] = False
tree_tops[:, -4:] = False

print(f"Detected tree tops: {tree_tops.sum():,}")

# Extract tree coordinates and heights
rows_idx, cols_idx = np.where(tree_tops)
tree_heights = chm[rows_idx, cols_idx]
x_coords = chm_transform.c + cols_idx * chm_transform.a + 0.5 * chm_transform.a
y_coords = chm_transform.f + rows_idx * chm_transform.e + 0.5 * chm_transform.e

# Estimate crown diameter from height (allometric: CD ≈ 1.2 * H^0.6)
crown_diameters = 1.2 * tree_heights ** 0.6

print(f"Height range: {tree_heights.min():.1f} - {tree_heights.max():.1f} m")
print(f"Mean height: {tree_heights.mean():.1f} ± {tree_heights.std():.1f} m")
print(f"Mean crown diameter: {crown_diameters.mean():.1f} ± {crown_diameters.std():.1f} m")

## Step 4: Allometric Biomass Estimation

Aboveground biomass (AGB) is calculated using the Jucker et al. (2017) model for gymnosperms:

$$\text{AGB} = 0.109 \times (H \times CD)^{1.79} \times 1.02$$

where:
- $H$ = tree height (m)
- $CD$ = crown diameter (m)
- 1.02 = bias correction for back-transformation (Snowdon 1991)

In [ ]:
def calculate_agb_jucker(height_m, crown_diameter_m, alpha=0.109, beta=1.79, bias_correction=1.02):
    """Calculate aboveground biomass using Jucker et al. (2017) gymnosperm model.
    
    Parameters
    ----------
    height_m : array-like
        Tree height in meters.
    crown_diameter_m : array-like
        Crown diameter in meters.
    alpha : float
        Intercept coefficient (default: 0.109 for gymnosperms).
    beta : float
        Scaling exponent (default: 1.79 for gymnosperms).
    bias_correction : float
        Back-transformation bias correction factor (default: 1.02).
    
    Returns
    -------
    agb_kg : array-like
        Aboveground biomass in kg.
    """
    height_m = np.asarray(height_m, dtype=float)
    crown_diameter_m = np.asarray(crown_diameter_m, dtype=float)
    agb_kg = alpha * (height_m * crown_diameter_m) ** beta * bias_correction
    return agb_kg

In [ ]:
# Example: verify with representative values
example_heights = np.array([5, 10, 15, 20, 25])
example_crowns = np.array([3, 5, 7, 8, 10])

example_agb = calculate_agb_jucker(example_heights, example_crowns)

print("Example AGB calculations (Jucker et al. 2017, gymnosperm model):")
print(f"{'Height (m)':>12} {'Crown D (m)':>12} {'AGB (kg)':>12}")
print("-" * 40)
for h, c, a in zip(example_heights, example_crowns, example_agb):
    print(f"{h:>12.1f} {c:>12.1f} {a:>12.1f}")

In [ ]:
# Calculate AGB for all segmented trees
agb_kg = calculate_agb_jucker(tree_heights, crown_diameters)
print(f"AGB range: {agb_kg.min():.1f} - {agb_kg.max():.1f} kg")
print(f"AGB mean: {agb_kg.mean():.1f} kg")
print(f"Total AGB: {agb_kg.sum():,.0f} kg ({agb_kg.sum()/1e6:.2f} Gg)")

## Step 5: Energy Content Calculation

Energy content is estimated using the Higher Heating Value (HHV) for *Pinus* species.

We use the mean HHV from nine estimates across five *Pinus* species:

$$\Delta H_R = 20.25 \pm 0.67 \text{ MJ/kg}$$

For each tree $i$:

$$E_i \text{ (MJ)} = \text{AGB}_i \text{ (kg)} \times \Delta H_R \text{ (MJ/kg)}$$

In [ ]:
# Higher Heating Values for Pinus species (MJ/kg)
# Compiled from literature
hhv_pinus = {
    'species': [
        'P. ponderosa', 'P. ponderosa',
        'P. contorta', 'P. contorta',
        'P. monticola',
        'P. sylvestris', 'P. sylvestris',
        'P. taeda', 'P. taeda',
    ],
    'hhv_mj_kg': [
        20.02, 20.37,  # P. ponderosa estimates
        20.10, 19.85,  # P. contorta estimates
        20.55,         # P. monticola
        20.54, 20.18,  # P. sylvestris estimates
        20.32, 20.35,  # P. taeda estimates
    ],
    'source': [
        'TBD', 'TBD',
        'TBD', 'TBD',
        'TBD',
        'TBD', 'TBD',
        'TBD', 'TBD',
    ]
}

hhv_df = pd.DataFrame(hhv_pinus)
print("Higher Heating Values for Pinus species:")
print(hhv_df.to_string(index=False))
print(f"\nMean HHV: {hhv_df['hhv_mj_kg'].mean():.2f} ± {hhv_df['hhv_mj_kg'].std():.2f} MJ/kg")

# Mean HHV for energy calculations
HHV_MEAN = hhv_df['hhv_mj_kg'].mean()  # ~20.25 MJ/kg
HHV_STD = hhv_df['hhv_mj_kg'].std()

In [ ]:
def calculate_energy_content(agb_kg, hhv_mj_per_kg=20.25):
    """Calculate energy content from biomass and higher heating value.
    
    Parameters
    ----------
    agb_kg : array-like
        Aboveground biomass in kg.
    hhv_mj_per_kg : float
        Higher heating value in MJ/kg (default: 20.25 for Pinus spp.).
    
    Returns
    -------
    energy_mj : array-like
        Energy content in MJ.
    """
    return np.asarray(agb_kg, dtype=float) * hhv_mj_per_kg

In [ ]:
# Calculate energy content for all trees
energy_mj = calculate_energy_content(agb_kg, HHV_MEAN)
print(f"Per-tree energy range: {energy_mj.min():.1f} - {energy_mj.max():.1f} MJ")
print(f"Per-tree energy mean: {energy_mj.mean():.1f} MJ")

## Step 6: Landscape Energy Census

Total landscape aboveground energy:

$$E_{\text{total}} = \sum_{i=1}^{n} E_i$$

Then normalize to energy density per unit area.

In [ ]:
# Build tree census DataFrame
n_trees = len(tree_heights)
trees_df = pd.DataFrame({
    'tree_id': range(1, n_trees + 1),
    'utm_easting': x_coords,
    'utm_northing': y_coords,
    'height_m': tree_heights,
    'crown_diameter_m': crown_diameters,
    'agb_kg': agb_kg,
    'energy_mj': energy_mj,
})

# Landscape energy summary
summary = landscape_energy_summary(trees_df, STUDY_AREA_KM2)

print("=== Landscape Energy Census ===")
print(f"Total trees:        {summary['n_trees']:,}")
print(f"Total energy:       {summary['total_energy_j']:.2e} J ({summary['total_energy_mj']:,.0f} MJ)")
print(f"Energy per km²:     {summary['energy_per_km2_j']:.2e} J/km²")
print(f"Energy per hectare: {summary['energy_per_ha_j']:.2e} J/ha")
print(f"Mean tree energy:   {summary['mean_tree_energy_mj']:.1f} MJ/tree")

# Save tree census CSV
trees_df.to_csv(TREE_CSV_PATH, index=False)
print(f"\nSaved: {TREE_CSV_PATH}")

# Display first rows
trees_df.head(10)

## Visualization

Rasterize per-tree energy values to obtain energy density distribution (J m⁻²).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# CHM
with rasterio.open(CHM_PATH) as src:
    show(src, ax=axes[0, 0], title="Canopy Height Model (0.5m)", cmap='YlGn')
    plt.colorbar(axes[0, 0].images[0], ax=axes[0, 0], label="Height (m)")

# Tree height distribution
axes[0, 1].hist(trees_df['height_m'], bins=50, color='forestgreen', edgecolor='darkgreen', alpha=0.8)
axes[0, 1].set_xlabel('Tree Height (m)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title(f'Tree Height Distribution (n={len(trees_df):,})')
axes[0, 1].axvline(trees_df['height_m'].mean(), color='red', linestyle='--', label=f"mean={trees_df['height_m'].mean():.1f} m")
axes[0, 1].legend()

# AGB distribution
axes[1, 0].hist(trees_df['agb_kg'], bins=50, color='sienna', edgecolor='saddlebrown', alpha=0.8)
axes[1, 0].set_xlabel('Aboveground Biomass (kg)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Per-Tree AGB Distribution')
axes[1, 0].axvline(trees_df['agb_kg'].mean(), color='red', linestyle='--', label=f"mean={trees_df['agb_kg'].mean():.1f} kg")
axes[1, 0].legend()

# Energy per tree (scatter on landscape)
sc = axes[1, 1].scatter(
    trees_df['utm_easting'], trees_df['utm_northing'],
    c=trees_df['energy_mj'], s=0.3, cmap='hot_r', alpha=0.6
)
axes[1, 1].set_xlabel('UTM Easting (m)')
axes[1, 1].set_ylabel('UTM Northing (m)')
axes[1, 1].set_title('Landscape Energy Field (MJ/tree)')
axes[1, 1].set_aspect('equal')
plt.colorbar(sc, ax=axes[1, 1], label='Energy (MJ)')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'energetics_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved figure: {OUTPUT_DIR / 'energetics_summary.png'}")

## References

- Jucker, T., et al. (2017). Allometric equations for integrating remote sensing imagery into forest monitoring programmes. *Global Change Biology*, 23(1), 177-190.
- Lin, H., et al. (2011). Earth's Critical Zone and hydropedology. *Hydrology and Earth System Sciences*, 15(12), 3895-3910.
- Snowdon, P. (1991). A ratio estimator for bias correction in logarithmic regressions. *Canadian Journal of Forest Research*, 21(5), 720-724.
- Swetnam, T.L. and Falk, D.A. (2014). Application of metabolic scaling theory to reduce error in local maxima tree segmentation from aerial LiDAR. *Forest Ecology and Management*, 323, 158-167.